In [4]:
import numpy as np
import pyvisa
import matplotlib.pyplot as plt
import time

In [2]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'GPIB0::3::INSTR', 'GPIB0::4::INSTR')


In [ ]:
bnc575 = rm.open_resource("GPIB0::3::INSTR")
# bnc575.query("*IDN?")

In [ ]:

bnc575.write(":Pulse0:Trigger:Mode Trig")
bnc575.write(":Pulse0:State off")
t = 2e-3
bnc575.write(f":Pulse0:Period {t}")
bnc575.write(":Pulse0:Mode NORm")
bnc575.write(":Pulse0:Bcounter 3")
a = bnc575.query(":Pulse0:Bcounter?")
print(a)
r = bnc575.query(":Pulse0:Mode?")
print(r)
bnc575.write(":Pulse1:State on")

bnc575.write(":Pulse1:MUX 1")
m = bnc575.query(":Pulse1:MUX?")
print(m)
bn = bin(int(m))[2:]
print(bn)
bnc575.write(":Pulse1:width 1e-3")
bnc575.write(":Pulse1:Output:Mode TTL")
# bnc575.write(":Pulse1:Output:AMP 6.0")
bnc575.write(":Pulse0:State on")
# bnc575.write("*Trg")

In [ ]:
bnc575.write("*Trg")

In [ ]:
binary = "111"
print(int("-1",10))

In [ ]:
b = 6
print(str(b))

In [ ]:
# version1
class BNC575:
    def __init__(self, visa_name, timeout = 5000):
        rm = pyvisa.ResourceManager()
        self.pyvisa = rm.open_resource(visa_name)
        self.pyvisa.timeout = timeout

    def clock_set(self, period, mode = "CONTINOUS", burstctr = -1):
        self.pyvisa.write(":Pulse0:Trigger:Mode Trig")
        self.pyvisa.write(f":Pulse0:Period {period}")
        if mode == "CONTINOUS":
            self.pyvisa.write(":Pulse0:Mode Normal")
        elif mode == "SINGLE":
            self.pyvisa.write(":Pulse0:Mode Single")
        elif mode == "BURST":
            self.pyvisa.write(":Pulse0:Mode Burst")
            if burstctr > 0:
                self.pyvisa.write(f":Pulse0:Bcounter {burstctr}")
            else:
                print("Burst number is invalid")
        else:
            print("Mode is invalid, the mode should be upper case. Valid Mode(CONTINOUS; SINGLE; BURST)")
            
        M = self.pyvisa.query(":Pulse0:Mode?")
        P = float(self.pyvisa.query(":Pulse0:Period?")[:-2])
        N = (self.pyvisa.query(":Pulse0:Bcounter?") if M == "BURS\r\n" else "NAN")
        print("---------------------------------------")
        print("The global clock is set to:\n" + "Mode: " + M + "Period: " + f"{P:.2e}s\n" + "Pulses number: " + N[:-1])
        print("---------------------------------------\n")

    def channel_set(self, channel:str, width, delay, AMP = 5.5, SYNC = "TO", MUX = "-1"):
        ch_dic = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}
        MUX_dic = {"A":1, "B":2, "C":4, "D":8, "E":16, "F":32, "G":64, "H":128}
        BMUX = int(MUX,2)
        self.pyvisa.write(f":Pulse{ch_dic[channel]}:State on")
        self.pyvisa.write(f":Pulse{ch_dic[channel]}:Width {width}")
        self.pyvisa.write(f":Pulse{ch_dic[channel]}:Delay {delay}")
        self.pyvisa.write(f":Pulse{ch_dic[channel]}:Output:Mode TTL")
        # self.pyvisa.write(f":Pulse{ch_dic[channel]}:Output:AMP {AMP}")
        self.pyvisa.write(f":Pulse{ch_dic[channel]}:SYNC {SYNC}")
        self.pyvisa.write(f":Pulse{ch_dic[channel]}:CMode Normal")
        if 0 < BMUX < 256:
            self.pyvisa.write(f":Pulse{ch_dic[channel]}:MUX {BMUX}")
        else:
            self.pyvisa.write(f":Pulse{ch_dic[channel]}:MUX {MUX_dic[channel]}")
            
        W = float(self.pyvisa.query(f":Pulse{ch_dic[channel]}:Width?")[:-2])
        D = float(self.pyvisa.query(f":Pulse{ch_dic[channel]}:Delay?")[:-2])
        # A = self.pyvisa.query(f":Pulse{ch_dic[channel]}:Output:AMP?")
        S = self.pyvisa.query(f":Pulse{ch_dic[channel]}:SYNC?")
        M = bin(int(self.pyvisa.query(f":Pulse{ch_dic[channel]}:MUX?")))[2:]
        print("---------------------------------------")
        print(f"The channel{channel} is set to:\n" + "Width: " + f"{W:.2e}s\n" + "Delay: " + f"{D:.2e}s\n"  + "Synchronize to " + S + "Output timers:\nHGFEDCBA\n" + M.zfill(8))
        print("---------------------------------------\n")
    
    def start_pulses(self):
        self.pyvisa.write(":Pulse0:State on")
        self.pyvisa.write("*Trg")
        
    def rearm_all(self):
        self.pyvisa.write("*Arm")


In [5]:
test = BNC575("GPIB0::3::INSTR")
test.disarm_all()

In [6]:
test.clock_set(period=2e-3, mode="BURST", burstctr=5)
test.channel_set(channel="A",width=1e-3,delay = 1e-3, MUX="11111111")

---------------------------------------
The global clock is set to:
Mode: BURS
Period: 2.00e-03s
Pulses number: ?3
---------------------------------------

---------------------------------------
The channel A is set to:
Width: 1.0000e-03s
Delay: 1.0000e-03s
Synchronize to: T0

Output timers:
HGFEDCBA
11111111
---------------------------------------



In [8]:
# version3 modified by ChatGPT
import pyvisa

class BNC575:
    def __init__(self, visa_name, timeout=5000):
        rm = pyvisa.ResourceManager()
        self.pyvisa = rm.open_resource(visa_name)
        self.pyvisa.timeout = timeout

    def clock_set(self, period, mode="CONTINOUS", burstctr=1):
        self.pyvisa.write(":Pulse0:Trigger:Mode Trig")
        self.pyvisa.write(f":Pulse0:Period {period}")
        if mode == "CONTINOUS":
            self.pyvisa.write(":Pulse0:Mode Normal")
        elif mode == "SINGLE":
            self.pyvisa.write(":Pulse0:Mode Single")
        elif mode == "BURST":
            self.pyvisa.write(":Pulse0:Mode Burst")
            if burstctr > 0:
                self.pyvisa.write(f":Pulse0:Bcounter {burstctr}")
            else:
                print("Burst number is invalid")
        else:
            print("Mode is invalid. Valid modes: CONTINOUS, SINGLE, BURST")
        
        M = self.pyvisa.query(":Pulse0:Mode?")[:-2]
        P = float(self.pyvisa.query(":Pulse0:Period?")[:-2])
        if M =="BURS":
            N = self.pyvisa.query(":Pulse0:Bcounter?")
        else:
            N = "NAN"
        print(N)
        
        print("---------------------------------------")
        print("The global clock is set to:")
        print(f"Mode: {M}")
        print(f"Period: {P:.2e}s")
        print(f"Pulses number: {N}")
        print("---------------------------------------\n")

    def channel_set(self, channel: str, width, delay, SYNC="TO", MUX="-1"):
        ch_dic = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "H": 8}
        MUX_dic = {"A": 1, "B": 2, "C": 4, "D": 8, "E": 16, "F": 32, "G": 64, "H": 128}

        if channel not in ch_dic:
            print("Invalid channel.")
            return

        ch_num = ch_dic[channel]
        print(ch_num)

        self.pyvisa.write(f":Pulse{ch_num}:State on")
        self.pyvisa.write(f":Pulse{ch_num}:Width {width}")
        self.pyvisa.write(f":Pulse{ch_num}:Delay {delay}")
        self.pyvisa.write(f":Pulse{ch_num}:Output:Mode TTL")
        # self.pyvisa.write(f":Pulse{ch_num}:Output:AMP {AMP}")
        self.pyvisa.write(f":Pulse{ch_num}:SYNC {SYNC}")
        self.pyvisa.write(f":Pulse{ch_num}:CMode Normal")

        try:
            BMUX = int(MUX, 2)
        except ValueError:
            BMUX = -1

        if 0 < BMUX < 256:
            self.pyvisa.write(f":Pulse{ch_num}:MUX {BMUX}")
        else:
            self.pyvisa.write(f":Pulse{ch_num}:MUX {MUX_dic[channel]}")

        W = float(self.pyvisa.query(f":Pulse{ch_num}:Width?")[:-2])
        D = float(self.pyvisa.query(f":Pulse{ch_num}:Delay?")[:-2])
        S = self.pyvisa.query(f":Pulse{ch_num}:SYNC?")[:-2]
        M = bin(int(self.pyvisa.query(f":Pulse{ch_num}:MUX?")))[2:]

        print("---------------------------------------")
        print(f"The channel {channel} is set to:")
        print(f"Width: {W:.4e}s")
        print(f"Delay: {D:.4e}s")
        print(f"Synchronize to: {S}")
        print("Output timers:\nHGFEDCBA")
        print(M.zfill(8))
        print("---------------------------------------\n")

    def start_pulses(self):
        self.pyvisa.write(":Pulse0:State on")
        self.pyvisa.write("*Trg")

    def rearm_all(self):
        self.pyvisa.write("*Arm")

    def disarm_all(self):
        self.pyvisa.write(":Pulse0:State off")

In [ ]:
int("-1",2)

In [ ]:
a = "0\r\n"
b = a.strip()
print(a)
print(b)
print(a)
if b == "0\r\n":
    print(1)
elif b == "0":
    print(2)

In [8]:
# my version, version4
import pyvisa
class BNC575:
    def __init__(self, visa_name, timeout = 5000):
        rm = pyvisa.ResourceManager()
        self.pyvisa = rm.open_resource(visa_name)
        self.pyvisa.timeout = timeout

    def clock_set(self, period, mode = "CONTINOUS", burstctr = -1):
        self.pyvisa.write(":Pulse0:Trigger:Mode Trig")
        self.pyvisa.write(f":Pulse0:Period {period}")
        if mode == "CONTINOUS":
            self.pyvisa.write(":Pulse0:Mode Normal")
        elif mode == "SINGLE":
            self.pyvisa.write(":Pulse0:Mode Single")
        elif mode == "BURST":
            self.pyvisa.write(":Pulse0:Mode Burst")
            if burstctr > 0:
                self.pyvisa.write(f":Pulse0:Bcounter {burstctr}")
            else:
                print("Burst number is invalid")
        else:
            print("Mode is invalid, the mode should be upper case. Valid Mode(CONTINOUS; SINGLE; BURST)")
            
        M = self.pyvisa.query(":Pulse0:Mode?")[:-2]
        P = float(self.pyvisa.query(":Pulse0:Period?")[:-2])
        if M == "BURS":
            try:
                N = self.pyvisa.query(":Pulse0:Bcounter?").strip()
            except pyvisa.errors.VisaIOError:
                N = "Query error"
        else:
            N = "N/A"
            
        print("---------------------------------------")
        print("The global clock is set to:")
        print(f"Mode: {M}")
        print(f"Period: {P:.2e}s")
        print(f"Pulses number: {N}")
        print("---------------------------------------")

    def channel_set(self, channel: str, width, delay, SYNC="TO", MUX="-1"):
        ch_dic = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}
        MUX_dic = {"A":1, "B":2, "C":4, "D":8, "E":16, "F":32, "G":64, "H":128}
        BMUX = int(MUX,2)
        if channel not in ch_dic:
            print("Invalid channel.")
            return

        ch_num = ch_dic[channel]
        
        self.pyvisa.write(f":Pulse{ch_num}:State on")
        self.pyvisa.write(f":Pulse{ch_num}:Width {width}")
        self.pyvisa.query("*OPC?")
        # time.sleep(0.002)
        self.pyvisa.write(f":Pulse{ch_num}:Delay {delay}")
        # time.sleep(0.002)
        self.pyvisa.query("*OPC?")
        self.pyvisa.write(f":Pulse{ch_num}:Output:Mode TTL")
        # time.sleep(0.002)
        self.pyvisa.query("*OPC?")
		# self.pyvisa.write(f":Pulse{ch_num}:Output:AMP {AMP}")
        self.pyvisa.write(f":Pulse{ch_num}:SYNC {SYNC}")
        # time.sleep(0.002)
        self.pyvisa.query("*OPC?")
        self.pyvisa.write(f":Pulse{ch_num}:CMode Normal")
        if 0 < BMUX < 256:
            self.pyvisa.write(f":Pulse{ch_dic[channel]}:MUX {BMUX}")
            # time.sleep(0.002)
            self.pyvisa.query("*OPC?")
        else:
            self.pyvisa.write(f":Pulse{ch_dic[channel]}:MUX {MUX_dic[channel]}")
            # time.sleep(0.002)
            self.pyvisa.query("*OPC?")
        	
        W = float(self.pyvisa.query(f":Pulse{ch_dic[channel]}:Width?")[:-2])
        # time.sleep(0.002)
        self.pyvisa.query("*OPC?")
        D = float(self.pyvisa.query(f":Pulse{ch_dic[channel]}:Delay?")[:-2])
        # time.sleep(0.002)
        self.pyvisa.query("*OPC?")
        # A = self.pyvisa.query(f":Pulse{ch_dic[channel]}:Output:AMP?")
        S = self.pyvisa.query(f":Pulse{ch_dic[channel]}:SYNC?")[:-2]
        # time.sleep(0.002)
        self.pyvisa.query("*OPC?")
        M = bin(int(self.pyvisa.query(f":Pulse{ch_dic[channel]}:MUX?")))[2:]
        # time.sleep(0.002)
        self.pyvisa.query("*OPC?")
        
        print("---------------------------------------")
        print(f"The channel {channel} is set to:")
        print(f"Width: {W:.4e}s")
        print(f"Delay: {D:.4e}s")
        print(f"Synchronize to: {S}")
        print("Output timers:\nHGFEDCBA")
        print(M.zfill(8))
        print("---------------------------------------")

    def start_pulses(self):
        self.pyvisa.write(":Pulse0:State on")
        self.pyvisa.write("*Trg")
        
    def rearm_all(self):
        self.pyvisa.write("*Arm")

    def disarm_all(self):
        self.pyvisa.write(":Pulse0:State off")

In [9]:
test = BNC575("GPIB0::9::INSTR")
test.disarm_all()

In [10]:
test.disarm_all()
test.clock_set(period=2e-3, mode="BURST", burstctr=5)
test.channel_set(channel="A",width=1e-3,delay = 1e-3, MUX="11111111")

---------------------------------------
The global clock is set to:
Mode: BURS
Period: 2.00e-03s
Pulses number: 5
---------------------------------------
---------------------------------------
The channel A is set to:
Width: 1.0000e-03s
Delay: 1.0000e-03s
Synchronize to: T0
Output timers:
HGFEDCBA
11111111
---------------------------------------
